# EP13. MCP 레지스트리와 거버넌스

## 학습 목표

1. **중앙 레지스트리**의 필요성과 설계 원칙을 이해한다
2. **Pydantic 데이터 모델**로 레지스트리 스키마를 정의하고 **FastMCP**로 레지스트리 서버를 구축한다
3. **Human-in-the-Loop** 승인 워크플로우와 **RBAC 접근 제어**를 구현한다
4. **Langfuse**로 MCP 서버 사용량을 모니터링하고 운영 가시성을 확보한다

## AI 가이드

> 막히는 부분은 Claude에게 물어보세요.
> 예시 프롬프트:
> - "MCP 레지스트리에서 태그 기반 디스커버리를 구현하는 방법을 알려줘"
> - "Human-in-the-Loop 승인 워크플로우의 상태 전이를 설명해줘"

**사전 요구사항**: EP12 수강, Python 기본 문법, Pydantic 기본 이해

**예상 소요 시간**: 75분

**필요한 API 키**:
- `ANTHROPIC_API_KEY`
- `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY`

---
## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install fastmcp langchain-anthropic langfuse pydantic python-dotenv --quiet

---
## 섹션 2. 라이브러리 로드 + API 키 설정

In [ ]:
import os
import json
import uuid
from datetime import datetime, timedelta
from enum import Enum
from dotenv import load_dotenv

# .env 파일에서 API 키 로드
load_dotenv()

# API 키 확인
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY가 설정되지 않았습니다"
assert os.getenv("LANGFUSE_PUBLIC_KEY"), "LANGFUSE_PUBLIC_KEY가 설정되지 않았습니다"

print("API 키 로드 완료 ✓")

---
## 섹션 3. Pinterest 사례 분석

### Pinterest의 MCP 도입 사례

Pinterest는 내부 데이터 인프라를 **도메인별 MCP 서버**로 분리하여 관리합니다.

| 도메인 | MCP 서버 | 주요 Tool |
|--------|---------|----------|
| 데이터 분석 | Presto MCP | `run_query`, `get_schema` |
| 배치 처리 | Spark MCP | `submit_job`, `job_status` |
| 워크플로우 | Airflow MCP | `trigger_dag`, `list_dags` |
| 이벤트 | Kafka MCP | `produce`, `consume_latest` |

**핵심**: 각 도메인 팀이 자기 MCP 서버를 소유하고, 중앙 레지스트리가 카탈로그 역할을 합니다.

In [ ]:
# Pinterest 사례: 도메인별 MCP 서버 아키텍처 (개념 코드)

# 실제 Pinterest처럼 도메인별 서버를 정의하는 구조
pinterest_architecture = {
    "presto-mcp": {
        "team": "Data Platform",
        "tools": ["run_query", "get_schema", "list_tables"],
        "transport": "http+sse",
        "users": 45,
        "calls_per_day": 12000
    },
    "spark-mcp": {
        "team": "ML Platform",
        "tools": ["submit_job", "job_status", "cancel_job"],
        "transport": "http+sse",
        "users": 20,
        "calls_per_day": 3500
    },
    "airflow-mcp": {
        "team": "Data Engineering",
        "tools": ["trigger_dag", "list_dags", "dag_status"],
        "transport": "http+sse",
        "users": 15,
        "calls_per_day": 890
    },
    "kafka-mcp": {
        "team": "Streaming Platform",
        "tools": ["produce", "consume_latest", "list_topics"],
        "transport": "http+sse",
        "users": 30,
        "calls_per_day": 25000
    }
}

print("=== Pinterest MCP 아키텍처 ===")
for server_id, info in pinterest_architecture.items():
    print(f"\n🔌 {server_id}")
    print(f"   팀: {info['team']}")
    print(f"   Tools: {', '.join(info['tools'])}")
    print(f"   사용자: {info['users']}명 | 일 호출: {info['calls_per_day']:,}회")

total_calls = sum(s["calls_per_day"] for s in pinterest_architecture.values())
print(f"\n📊 일일 총 호출: {total_calls:,}회")
print("\n→ 이 규모를 관리하려면 중앙 레지스트리가 필수!")
print("완료 ✓")

---
## 섹션 4. 레지스트리 데이터 모델 (Pydantic)

레지스트리가 관리할 핵심 데이터 구조를 Pydantic 모델로 정의합니다.

In [ ]:
from pydantic import BaseModel, Field

# --- 서버 상태 ---
class ServerStatus(str, Enum):
    ACTIVE = "active"
    INACTIVE = "inactive"
    MAINTENANCE = "maintenance"

# --- Tool 정보 ---
class ToolInfo(BaseModel):
    """MCP 서버가 제공하는 개별 Tool의 메타데이터"""
    name: str = Field(..., description="Tool 이름")
    description: str = Field(..., description="Tool 설명")
    risk_level: str = Field(default="LOW", description="위험 수준: LOW/MEDIUM/HIGH/CRITICAL")
    parameters: dict = Field(default_factory=dict, description="파라미터 스키마")

# --- MCP 서버 레코드 ---
class MCPServerRecord(BaseModel):
    """레지스트리에 등록된 MCP 서버 정보"""
    server_id: str = Field(..., description="고유 식별자")
    name: str = Field(..., description="표시 이름")
    description: str = Field(..., description="서버 설명")
    version: str = Field(default="1.0.0", description="서버 버전 (semver)")
    status: ServerStatus = Field(default=ServerStatus.ACTIVE)
    owner: str = Field(..., description="소유 팀/담당자")
    tags: list[str] = Field(default_factory=list, description="검색용 태그")
    tools: list[ToolInfo] = Field(default_factory=list, description="제공 Tool 목록")
    endpoint: str = Field(..., description="연결 주소")
    registered_at: datetime = Field(default_factory=datetime.now)
    last_heartbeat: datetime | None = Field(default=None)

# --- 승인 요청 ---
class ApprovalStatus(str, Enum):
    PENDING = "pending"
    APPROVED = "approved"
    REJECTED = "rejected"
    EXPIRED = "expired"

class ApprovalRequest(BaseModel):
    """Human-in-the-Loop 승인 요청"""
    request_id: str = Field(..., description="요청 ID")
    user: str = Field(..., description="요청자")
    server_id: str = Field(..., description="대상 서버")
    tool_name: str = Field(..., description="대상 Tool")
    risk_level: str = Field(..., description="위험 수준")
    status: ApprovalStatus = Field(default=ApprovalStatus.PENDING)
    requested_at: datetime = Field(default_factory=datetime.now)
    expires_at: datetime = Field(default_factory=lambda: datetime.now() + timedelta(minutes=30))
    reviewer: str | None = Field(default=None)
    reason: str | None = Field(default=None, description="거부 사유")

# --- 접근 제어 권한 ---
class Permission(BaseModel):
    """RBAC 접근 제어 권한"""
    role: str = Field(..., description="역할 (developer, analyst, admin)")
    server_id: str = Field(..., description="대상 서버")
    allowed_tools: list[str] = Field(default_factory=list, description="허용 Tool 목록")
    max_calls_per_hour: int = Field(default=100, description="시간당 최대 호출")
    risk_level_max: str = Field(default="MEDIUM", description="최대 허용 위험 수준")

# 모델 테스트
test_tool = ToolInfo(name="query_products", description="상품 조회", risk_level="LOW")
test_server = MCPServerRecord(
    server_id="test-mcp",
    name="Test MCP Server",
    description="테스트용 서버",
    owner="platform_team",
    endpoint="stdio://test_server.py",
    tags=["test", "demo"],
    tools=[test_tool]
)

print("=== 데이터 모델 테스트 ===")
print(f"서버: {test_server.name} (v{test_server.version})")
print(f"상태: {test_server.status.value}")
print(f"Tools: {[t.name for t in test_server.tools]}")
print(f"태그: {test_server.tags}")
print(f"\nJSON 출력:")
print(test_server.model_dump_json(indent=2)[:300] + "...")
print("\n데이터 모델 정의 완료 ✓")

---
## 섹션 5. 레지스트리 서버 구현

레지스트리 자체를 **FastMCP 서버**로 구현합니다.
이렇게 하면 에이전트가 MCP 프로토콜로 레지스트리와 통신할 수 있습니다.

In [ ]:
from fastmcp import FastMCP

# 레지스트리 서버 생성
registry = FastMCP("MCP Registry Server")

# 인메모리 저장소
_servers: dict[str, MCPServerRecord] = {}
_approvals: dict[str, ApprovalRequest] = {}
_policies: dict[str, list[Permission]] = {}
_call_log: list[dict] = []  # 호출 이력

print(f"레지스트리 서버 생성 완료: {registry.name}")
print("완료 ✓")

---
## 섹션 6. 서버 등록/조회/검색 API

레지스트리의 핵심 CRUD 기능을 MCP Tool로 구현합니다.

In [ ]:
# --- 서버 등록 ---
@registry.tool
def register_server(
    server_id: str,
    name: str,
    description: str,
    owner: str,
    endpoint: str,
    version: str = "1.0.0",
    tags: list[str] = []
) -> str:
    """새 MCP 서버를 레지스트리에 등록합니다."""
    if server_id in _servers:
        return f"❌ 이미 등록된 서버입니다: {server_id}"
    record = MCPServerRecord(
        server_id=server_id, name=name,
        description=description, owner=owner,
        endpoint=endpoint, version=version, tags=tags
    )
    _servers[server_id] = record
    return f"✅ {name} 등록 완료 (ID: {server_id}, v{version})"

# --- 서버 조회 ---
@registry.tool
def get_server(server_id: str) -> str:
    """특정 MCP 서버의 상세 정보를 조회합니다."""
    if server_id not in _servers:
        return f"❌ 서버를 찾을 수 없습니다: {server_id}"
    return _servers[server_id].model_dump_json(indent=2)

# --- 서버 검색 (디스커버리) ---
@registry.tool
def discover_servers(tag: str = "", status: str = "active") -> str:
    """태그와 상태로 MCP 서버를 검색합니다."""
    results = [
        s for s in _servers.values()
        if (not tag or tag in s.tags)
        and s.status.value == status
    ]
    if not results:
        return "검색 결과가 없습니다."
    return json.dumps(
        [{"id": s.server_id, "name": s.name, "version": s.version,
          "tags": s.tags, "tools": len(s.tools)} for s in results],
        ensure_ascii=False, indent=2
    )

# --- Tool 목록 조회 ---
@registry.tool
def list_tools(server_id: str) -> str:
    """특정 서버가 제공하는 Tool 목록을 반환합니다."""
    if server_id not in _servers:
        return f"❌ 서버를 찾을 수 없습니다: {server_id}"
    tools = _servers[server_id].tools
    return json.dumps([t.model_dump() for t in tools], ensure_ascii=False, indent=2)

# --- Tool 등록 (서버에 Tool 추가) ---
@registry.tool
def add_tool_to_server(
    server_id: str, tool_name: str,
    description: str, risk_level: str = "LOW"
) -> str:
    """서버에 Tool 메타데이터를 등록합니다."""
    if server_id not in _servers:
        return f"❌ 서버를 찾을 수 없습니다: {server_id}"
    tool = ToolInfo(name=tool_name, description=description, risk_level=risk_level)
    _servers[server_id].tools.append(tool)
    return f"✅ {tool_name} Tool이 {server_id}에 등록 완료 (위험: {risk_level})"

# --- Heartbeat ---
@registry.tool
def heartbeat(server_id: str) -> str:
    """서버 상태를 갱신합니다 (헬스 체크)."""
    if server_id not in _servers:
        return f"❌ 서버를 찾을 수 없습니다: {server_id}"
    _servers[server_id].last_heartbeat = datetime.now()
    return f"💓 {server_id} heartbeat 갱신 완료"

# --- 카탈로그 리소스 ---
@registry.resource("registry://catalog")
def get_catalog() -> str:
    """전체 서버 카탈로그를 반환합니다."""
    return json.dumps(
        {sid: {"name": s.name, "status": s.status.value, "version": s.version}
         for sid, s in _servers.items()},
        ensure_ascii=False, indent=2
    )

print("레지스트리 CRUD Tool 등록 완료 ✓")
print("  register_server, get_server, discover_servers")
print("  list_tools, add_tool_to_server, heartbeat")
print("  resource: registry://catalog")

In [ ]:
# 서버 등록 테스트
print("=== 서버 등록 테스트 ===")

# Pinterest 스타일 서버 등록
print(register_server(
    server_id="presto-mcp", name="Presto MCP Server",
    description="SQL 분석 쿼리 실행", owner="data_platform",
    endpoint="http://presto-mcp:8080/mcp",
    tags=["database", "analytics", "sql"]
))

print(register_server(
    server_id="spark-mcp", name="Spark MCP Server",
    description="대규모 배치 처리", owner="ml_platform",
    endpoint="http://spark-mcp:8080/mcp",
    tags=["batch", "ml", "processing"]
))

print(register_server(
    server_id="airflow-mcp", name="Airflow MCP Server",
    description="워크플로우 관리", owner="data_engineering",
    endpoint="http://airflow-mcp:8080/mcp",
    tags=["workflow", "orchestration", "scheduling"]
))

print(register_server(
    server_id="hr-mcp", name="HR MCP Server",
    description="인사 데이터 관리 (민감)", owner="hr_team",
    endpoint="http://hr-mcp:8080/mcp",
    tags=["hr", "sensitive", "personnel"]
))

# Tool 추가
print("\n=== Tool 등록 ===")
print(add_tool_to_server("presto-mcp", "run_query", "SQL 쿼리 실행", "MEDIUM"))
print(add_tool_to_server("presto-mcp", "get_schema", "테이블 스키마 조회", "LOW"))
print(add_tool_to_server("spark-mcp", "submit_job", "Spark 작업 제출", "HIGH"))
print(add_tool_to_server("spark-mcp", "job_status", "작업 상태 조회", "LOW"))
print(add_tool_to_server("airflow-mcp", "trigger_dag", "DAG 트리거", "HIGH"))
print(add_tool_to_server("airflow-mcp", "list_dags", "DAG 목록 조회", "LOW"))
print(add_tool_to_server("hr-mcp", "get_employee", "직원 정보 조회", "HIGH"))
print(add_tool_to_server("hr-mcp", "delete_employee", "직원 레코드 삭제", "CRITICAL"))

# 검색 테스트
print("\n=== 서버 검색 (태그: database) ===")
print(discover_servers(tag="database"))

print("\n=== 전체 서버 검색 ===")
print(discover_servers())

print("\n완료 ✓")

---
## 섹션 7. Human-in-the-Loop 승인 로직

고위험 Tool 호출에 대해 사람의 승인을 요구하는 워크플로우를 구현합니다.

**위험 수준별 승인 정책**:
- `LOW`: 자동 승인 (읽기 전용)
- `MEDIUM`: 팀 리드 승인
- `HIGH`: 관리자 승인
- `CRITICAL`: 다중 승인 (2인 이상)

In [ ]:
# --- 승인 요청 ---
@registry.tool
def request_approval(
    user: str, server_id: str,
    tool_name: str, risk_level: str
) -> str:
    """고위험 Tool 호출에 대한 승인을 요청합니다."""
    # 저위험은 자동 승인
    if risk_level == "LOW":
        _call_log.append({
            "user": user, "server_id": server_id,
            "tool": tool_name, "action": "auto_approved",
            "timestamp": datetime.now().isoformat()
        })
        return f"✅ 자동 승인 (저위험): {tool_name}"

    # 중위험 이상은 수동 승인 필요
    req_id = str(uuid.uuid4())[:8]
    req = ApprovalRequest(
        request_id=req_id, user=user,
        server_id=server_id, tool_name=tool_name,
        risk_level=risk_level
    )
    _approvals[req_id] = req
    _call_log.append({
        "user": user, "server_id": server_id,
        "tool": tool_name, "action": "approval_requested",
        "request_id": req_id, "risk": risk_level,
        "timestamp": datetime.now().isoformat()
    })
    return f"⏳ 승인 대기 중 (ID: {req_id}, 위험: {risk_level}, 만료: 30분)"

# --- 승인 처리 ---
@registry.tool
def approve_request(request_id: str, reviewer: str) -> str:
    """승인 요청을 승인합니다."""
    if request_id not in _approvals:
        return f"❌ 요청을 찾을 수 없습니다: {request_id}"
    req = _approvals[request_id]
    # 만료 확인
    if datetime.now() > req.expires_at:
        req.status = ApprovalStatus.EXPIRED
        return f"⏰ 승인 요청이 만료되었습니다: {request_id}"
    req.status = ApprovalStatus.APPROVED
    req.reviewer = reviewer
    _call_log.append({
        "request_id": request_id, "action": "approved",
        "reviewer": reviewer, "timestamp": datetime.now().isoformat()
    })
    return f"✅ 승인 완료: {req.tool_name} (by {reviewer})"

# --- 거부 처리 ---
@registry.tool
def reject_request(request_id: str, reviewer: str, reason: str = "") -> str:
    """승인 요청을 거부합니다."""
    if request_id not in _approvals:
        return f"❌ 요청을 찾을 수 없습니다: {request_id}"
    req = _approvals[request_id]
    req.status = ApprovalStatus.REJECTED
    req.reviewer = reviewer
    req.reason = reason
    _call_log.append({
        "request_id": request_id, "action": "rejected",
        "reviewer": reviewer, "reason": reason,
        "timestamp": datetime.now().isoformat()
    })
    return f"❌ 거부됨: {req.tool_name} (사유: {reason})"

# --- 승인 상태 확인 ---
@registry.tool
def check_approval(request_id: str) -> str:
    """승인 요청의 현재 상태를 확인합니다."""
    if request_id not in _approvals:
        return f"❌ 요청을 찾을 수 없습니다: {request_id}"
    req = _approvals[request_id]
    # 자동 만료 확인
    if req.status == ApprovalStatus.PENDING and datetime.now() > req.expires_at:
        req.status = ApprovalStatus.EXPIRED
    return json.dumps({
        "request_id": req.request_id,
        "status": req.status.value,
        "tool": req.tool_name,
        "risk": req.risk_level,
        "reviewer": req.reviewer,
        "reason": req.reason
    }, ensure_ascii=False, indent=2)

print("승인 워크플로우 Tool 등록 완료 ✓")
print("  request_approval, approve_request, reject_request, check_approval")

In [ ]:
# 승인 워크플로우 테스트
print("=== 승인 워크플로우 테스트 ===")

# 시나리오 1: 저위험 — 자동 승인
print("\n[시나리오 1] 저위험 Tool 호출")
result = request_approval("김개발", "presto-mcp", "get_schema", "LOW")
print(f"  결과: {result}")

# 시나리오 2: 고위험 — 수동 승인 필요
print("\n[시나리오 2] 고위험 Tool 호출")
result = request_approval("김개발", "spark-mcp", "submit_job", "HIGH")
print(f"  결과: {result}")

# 승인 ID 추출
pending_id = list(_approvals.keys())[-1]
print(f"  승인 ID: {pending_id}")

# 상태 확인
print(f"  상태: {check_approval(pending_id)}")

# 관리자가 승인
print(f"\n[관리자 승인]")
print(f"  {approve_request(pending_id, '박팀장')}")
print(f"  상태: {check_approval(pending_id)}")

# 시나리오 3: CRITICAL — 거부
print("\n[시나리오 3] CRITICAL Tool 호출 (거부)")
result = request_approval("이인턴", "hr-mcp", "delete_employee", "CRITICAL")
print(f"  결과: {result}")
critical_id = list(_approvals.keys())[-1]
print(f"  {reject_request(critical_id, '박팀장', '인턴은 삭제 권한 없음')}")

print("\n완료 ✓")

---
## 섹션 8. 접근 제어 정책 (RBAC)

역할(Role) 기반으로 MCP 서버와 Tool에 대한 접근 권한을 관리합니다.

| 역할 | 접근 가능 서버 | 최대 위험 수준 |
|------|-------------|------------|
| `developer` | presto, spark | MEDIUM |
| `analyst` | presto (읽기만) | LOW |
| `admin` | 모든 서버 | CRITICAL |

In [ ]:
# 위험 수준 순서 (비교용)
RISK_ORDER = {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "CRITICAL": 3}

# --- 정책 추가 ---
@registry.tool
def add_policy(
    role: str, server_id: str,
    allowed_tools: list[str],
    max_calls: int = 100,
    risk_max: str = "MEDIUM"
) -> str:
    """역할 기반 접근 정책을 추가합니다."""
    perm = Permission(
        role=role, server_id=server_id,
        allowed_tools=allowed_tools,
        max_calls_per_hour=max_calls,
        risk_level_max=risk_max
    )
    _policies.setdefault(role, []).append(perm)
    return f"✅ 정책 추가: {role} → {server_id} ({len(allowed_tools)} tools, 최대위험: {risk_max})"

# --- 접근 확인 ---
@registry.tool
def check_access(role: str, server_id: str, tool_name: str) -> str:
    """역할이 특정 Tool에 접근 가능한지 확인합니다."""
    perms = _policies.get(role, [])
    for p in perms:
        if p.server_id == server_id and tool_name in p.allowed_tools:
            return f"✅ 접근 허용: {role} → {server_id}/{tool_name} (최대 {p.max_calls_per_hour}회/시간)"
    return f"❌ 접근 거부: {role} → {server_id}/{tool_name}"

# --- 통합 게이트키퍼: 접근 제어 + 승인 워크플로우 ---
@registry.tool
def gatekeeper(
    user: str, role: str,
    server_id: str, tool_name: str
) -> str:
    """접근 제어와 승인 워크플로우를 통합한 게이트키퍼.
    1. RBAC 접근 확인 → 2. Tool 위험 수준 확인 → 3. 필요시 승인 요청"""
    # Step 1: RBAC 확인
    perms = _policies.get(role, [])
    matched = None
    for p in perms:
        if p.server_id == server_id and tool_name in p.allowed_tools:
            matched = p
            break
    if not matched:
        return f"❌ RBAC 거부: {role}은(는) {server_id}/{tool_name}에 접근 불가"

    # Step 2: Tool 위험 수준 확인
    server = _servers.get(server_id)
    if not server:
        return f"❌ 서버를 찾을 수 없습니다: {server_id}"
    tool = next((t for t in server.tools if t.name == tool_name), None)
    if not tool:
        return f"❌ Tool을 찾을 수 없습니다: {tool_name}"

    # 역할의 최대 위험 수준과 비교
    if RISK_ORDER.get(tool.risk_level, 0) > RISK_ORDER.get(matched.risk_level_max, 1):
        return f"❌ 위험 수준 초과: {tool_name}({tool.risk_level}) > {role} 최대({matched.risk_level_max})"

    # Step 3: 승인 필요 여부 결정
    return request_approval(user, server_id, tool_name, tool.risk_level)

print("접근 제어 Tool 등록 완료 ✓")
print("  add_policy, check_access, gatekeeper")

In [ ]:
# 접근 제어 테스트
print("=== RBAC 정책 설정 ===")

# 정책 추가
print(add_policy("developer", "presto-mcp", ["run_query", "get_schema"], 200, "MEDIUM"))
print(add_policy("developer", "spark-mcp", ["submit_job", "job_status"], 50, "HIGH"))
print(add_policy("analyst", "presto-mcp", ["get_schema"], 100, "LOW"))
print(add_policy("admin", "presto-mcp", ["run_query", "get_schema"], 500, "CRITICAL"))
print(add_policy("admin", "hr-mcp", ["get_employee", "delete_employee"], 50, "CRITICAL"))

print("\n=== 접근 확인 테스트 ===")
print(check_access("developer", "presto-mcp", "run_query"))
print(check_access("analyst", "presto-mcp", "run_query"))  # 거부
print(check_access("admin", "hr-mcp", "delete_employee"))

print("\n=== 게이트키퍼 통합 테스트 ===")
# developer가 presto 스키마 조회 (LOW → 자동 승인)
print(f"[developer→presto/get_schema] {gatekeeper('김개발', 'developer', 'presto-mcp', 'get_schema')}")

# developer가 spark 작업 제출 (HIGH → 승인 필요)
print(f"[developer→spark/submit_job] {gatekeeper('김개발', 'developer', 'spark-mcp', 'submit_job')}")

# analyst가 presto 쿼리 실행 시도 (RBAC 거부)
print(f"[analyst→presto/run_query] {gatekeeper('이분석', 'analyst', 'presto-mcp', 'run_query')}")

# admin이 HR 직원 삭제 (CRITICAL → 승인 필요)
print(f"[admin→hr/delete_employee] {gatekeeper('관리자', 'admin', 'hr-mcp', 'delete_employee')}")

print("\n완료 ✓")

---
## 섹션 9. 카탈로그 CLI 도구

레지스트리 상태를 한눈에 확인하는 CLI 스타일 도구를 구현합니다.

In [ ]:
# --- 카탈로그 요약 ---
@registry.tool
def catalog_summary() -> str:
    """레지스트리 전체 요약을 출력합니다."""
    total = len(_servers)
    active = sum(1 for s in _servers.values() if s.status == ServerStatus.ACTIVE)
    total_tools = sum(len(s.tools) for s in _servers.values())
    pending = sum(1 for a in _approvals.values() if a.status == ApprovalStatus.PENDING)

    lines = [
        "╔══════════════════════════════════════╗",
        "║     MCP Registry Dashboard           ║",
        "╠══════════════════════════════════════╣",
        f"║  총 서버: {total}개",
        f"║  활성 서버: {active}개",
        f"║  총 Tool 수: {total_tools}개",
        f"║  대기 중 승인: {pending}건",
        "╠══════════════════════════════════════╣",
        "║  서버 목록:",
    ]
    for sid, s in _servers.items():
        icon = "🟢" if s.status == ServerStatus.ACTIVE else "🔴"
        hb = "N/A"
        if s.last_heartbeat:
            delta = (datetime.now() - s.last_heartbeat).seconds
            hb = f"{delta}초 전"
        lines.append(f"║  {icon} {s.name} v{s.version}")
        lines.append(f"║     ID: {sid} | 소유: {s.owner}")
        lines.append(f"║     Tools: {len(s.tools)}개 | HB: {hb}")
    lines.append("╚══════════════════════════════════════╝")
    return "\n".join(lines)

# --- 서버 상태 점검 ---
@registry.tool
def health_check() -> str:
    """전체 서버의 헬스 상태를 확인합니다."""
    results = []
    for sid, s in _servers.items():
        if s.status != ServerStatus.ACTIVE:
            results.append(f"🔴 {sid}: {s.status.value}")
        elif s.last_heartbeat is None:
            results.append(f"🟡 {sid}: heartbeat 미수신")
        elif (datetime.now() - s.last_heartbeat).seconds > 600:
            results.append(f"🟠 {sid}: heartbeat {(datetime.now() - s.last_heartbeat).seconds}초 전 (비활성 의심)")
        else:
            results.append(f"🟢 {sid}: 정상")
    return "\n".join(results)

print("카탈로그 CLI 도구 등록 완료 ✓")
print("  catalog_summary, health_check")

In [ ]:
# 카탈로그 대시보드 출력
print(catalog_summary())

# Heartbeat 보내기
print("\n=== Heartbeat 전송 ===")
print(heartbeat("presto-mcp"))
print(heartbeat("spark-mcp"))

# 헬스 체크
print("\n=== 헬스 체크 ===")
print(health_check())

print("\n완료 ✓")

---
## 섹션 10. Langfuse 통합 (사용량 추적)

MCP Tool 호출을 Langfuse에 기록하여 사용량, 지연 시간, 오류율을 모니터링합니다.

In [ ]:
from langfuse import Langfuse
import time

# Langfuse 클라이언트 초기화
langfuse = Langfuse()

def trace_tool_call(
    server_id: str, tool_name: str,
    user: str, params: dict, result: str
):
    """MCP Tool 호출을 Langfuse에 기록합니다."""
    trace = langfuse.trace(
        name=f"mcp_{server_id}_{tool_name}",
        user_id=user,
        metadata={
            "server_id": server_id,
            "tool": tool_name,
            "source": "mcp_registry"
        }
    )
    # 실행 span
    span = trace.span(name="tool_execution", input=params)
    span.end(output={"result_preview": result[:200], "result_length": len(result)})
    # 비용 추적 generation (mock)
    trace.generation(
        name="registry_overhead",
        model="registry-v1",
        input=f"gatekeeper check for {server_id}/{tool_name}",
        output=result[:100],
        usage={"input": len(str(params)), "output": len(result)}
    )
    return trace

# 사용량 리포트 Tool
@registry.tool
def usage_report(server_id: str = "") -> str:
    """서버별 사용량 리포트를 생성합니다 (mock 데이터)."""
    mock_stats = {
        "presto-mcp": {"calls": 1523, "avg_latency_ms": 230, "errors": 12, "top_user": "data_team"},
        "spark-mcp": {"calls": 891, "avg_latency_ms": 450, "errors": 3, "top_user": "ml_team"},
        "airflow-mcp": {"calls": 456, "avg_latency_ms": 180, "errors": 1, "top_user": "de_team"},
        "hr-mcp": {"calls": 45, "avg_latency_ms": 120, "errors": 0, "top_user": "hr_admin"},
    }
    if server_id and server_id in mock_stats:
        return json.dumps({server_id: mock_stats[server_id]}, ensure_ascii=False, indent=2)
    return json.dumps(mock_stats, ensure_ascii=False, indent=2)

print("Langfuse 연동 설정 완료 ✓")
print("  trace_tool_call 함수, usage_report Tool")

In [ ]:
# Langfuse 트레이싱 테스트
print("=== Langfuse 트레이싱 테스트 ===")

# 여러 Tool 호출 시뮬레이션
calls = [
    ("presto-mcp", "get_schema", "김개발", {"table": "products"}),
    ("presto-mcp", "run_query", "김개발", {"sql": "SELECT COUNT(*) FROM products"}),
    ("spark-mcp", "job_status", "이엔지", {"job_id": "spark-001"}),
    ("airflow-mcp", "list_dags", "박운영", {}),
]

for server_id, tool_name, user, params in calls:
    # 게이트키퍼 없이 직접 트레이스 (테스트용)
    result = f"Mock result for {tool_name}"
    trace = trace_tool_call(server_id, tool_name, user, params, result)
    print(f"  📡 {user} → {server_id}/{tool_name} | trace_id: {trace.id[:12]}...")

langfuse.flush()
print("\nLangfuse 트레이스 전송 완료 ✓")
print("Langfuse 대시보드에서 mcp_* 트레이스를 확인하세요.")

# 사용량 리포트
print("\n=== 사용량 리포트 ===")
print(usage_report())

---
## 섹션 11. Exercise

### Exercise 1: 자동 비활성화 기능

**목표**: heartbeat 호출이 10분 이상 없는 서버를 자동으로 `INACTIVE` 처리

**구현할 것**:
1. `deactivate_stale_servers(threshold_minutes: int = 10)` — 일정 시간 heartbeat 없는 서버 비활성화
2. 비활성화 시 `_call_log`에 이벤트 기록
3. `reactivate_server(server_id: str)` — 서버 재활성화
4. 비활성화된 서버의 Tool 호출을 `gatekeeper`에서 차단

In [ ]:
# TODO: Exercise 1 구현

@registry.tool
def deactivate_stale_servers(threshold_minutes: int = 10) -> str:
    """heartbeat가 일정 시간 없는 서버를 비활성화합니다."""
    # 힌트: _servers를 순회하며 last_heartbeat 확인
    # 힌트: (datetime.now() - last_heartbeat).total_seconds() / 60 > threshold_minutes
    # 힌트: 상태를 ServerStatus.INACTIVE로 변경
    # 힌트: _call_log에 기록 추가
    pass  # 여기에 구현하세요

@registry.tool
def reactivate_server(server_id: str) -> str:
    """비활성화된 서버를 재활성화합니다."""
    # 힌트: 서버 상태를 ACTIVE로 변경
    # 힌트: last_heartbeat을 현재 시간으로 갱신
    pass  # 여기에 구현하세요

print("Exercise 1: deactivate_stale_servers, reactivate_server 구현하세요")

### Exercise 2: 승인 대시보드

**목표**: 미승인 요청을 조회하고 일괄 처리하는 도구 구축

**구현할 것**:
1. `list_pending_approvals()` — 대기 중 승인 요청 목록
2. `bulk_approve(request_ids: list[str], reviewer: str)` — 일괄 승인
3. `cleanup_expired()` — 만료된 요청 자동 정리
4. `approval_stats()` — 승인/거부/만료 통계

In [ ]:
# TODO: Exercise 2 구현

@registry.tool
def list_pending_approvals() -> str:
    """대기 중인 승인 요청 목록을 반환합니다."""
    # 힌트: _approvals에서 status == PENDING인 것만 필터
    pass  # 여기에 구현하세요

@registry.tool
def bulk_approve(request_ids: list[str], reviewer: str) -> str:
    """여러 승인 요청을 일괄 승인합니다."""
    # 힌트: request_ids를 순회하며 approve_request 호출
    pass  # 여기에 구현하세요

@registry.tool
def cleanup_expired() -> str:
    """만료된 승인 요청을 정리합니다."""
    # 힌트: expires_at < datetime.now()인 PENDING 요청을 EXPIRED로 변경
    pass  # 여기에 구현하세요

@registry.tool
def approval_stats() -> str:
    """승인/거부/만료 통계를 반환합니다."""
    # 힌트: _approvals의 status별 count
    pass  # 여기에 구현하세요

print("Exercise 2: list_pending_approvals, bulk_approve, cleanup_expired, approval_stats 구현하세요")

---
## 마무리

### 오늘 배운 것

- MCP 서버가 많아지면 **중앙 레지스트리**가 필수 (디스커버리, 상태 관리)
- **Pinterest 사례**: 도메인별 MCP 서버 분리 + 중앙 카탈로그 패턴
- **Pydantic 데이터 모델**로 서버, Tool, 승인, 권한을 체계적으로 정의
- **FastMCP**로 레지스트리 자체를 MCP 서버로 구축 (레지스트리 = MCP 서버)
- **Human-in-the-Loop**: 위험 수준에 따라 자동 승인 / 수동 승인 분기
- **RBAC 접근 제어**: 역할별 서버 · Tool · 위험 수준 권한 관리
- **Langfuse 통합**: Tool 호출 트레이싱으로 사용량 · 지연 · 에러 모니터링

### 다음 에피소드

**EP14. MCP 서버 간 연쇄 호출** — 멀티 에이전트 오케스트레이션